# tw-market-data-lake — Data Quality Audit v0.1.0（Phase A，唯讀）

- **唯讀**：不寫入、搬移、刪除 DATA_ROOT 下任何資料集、manifest、pointer、lock、staging、quarantine、runs 或既有資料夾；不呼叫任何行情 API；不需要 FinLab / 富邦憑證。
- 只讀各 dataset `_manifest/published.json` 指向的版本與其 `partitions.parquet` 列出的檔案。
- 輸出先寫到 Colab 本機 `/content/tw_lake_audits/<audit_id>/`；`PUBLISH_TO_DRIVE=True` 時，完成後才**複製**到 `_tw-market-data-lake/audits/<audit_id>/`（新資料夾，不覆蓋任何既有路徑），並 flush Drive。
- 全量（8,839 個 5 分K分區）約需讀 1.5 GB，Colab 上預估 20–60 分鐘；可先用 `SYMBOLS` 小範圍試跑。

In [ ]:
# 1. 參數
SYMBOLS = ""              # 例 "2330,0050,8299"；空白 = 全部 5m 分區
MAX_PARTITIONS = None     # 例 200；None = 不限
DEEP_VERIFY = False       # True: 所有已發布檔案 SHA256（最慢、最完整）
SHA_SAMPLE = 20           # 每個 dataset 隨機抽 N 個檔案做 SHA256
PUBLISH_TO_DRIVE = False  # True: 完成後把報表複製到 Drive 的 _tw-market-data-lake/audits/<audit_id>/

REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
PROJECT_DIR = "/content/tw-market-data-lake"
OUT_DIR = "/content/tw_lake_audits"

In [ ]:
# 2. 掛載 Google Drive（只讀取；稽核程式本身不寫入 DATA_ROOT）
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull 指定 branch，記錄 HEAD SHA
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
SHA = subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip()
os.environ["TW_LAKE_GIT_SHA"] = SHA
print("branch:", BRANCH, "HEAD:", SHA)

In [ ]:
# 4. 依賴（Colab 內建 pandas / pyarrow；只補 pyyaml / tqdm）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")

In [ ]:
# 5. 執行唯讀稽核
import sys, subprocess
args = [sys.executable, "-m", "tw_lake.audit.cli", "--data-root", DATA_ROOT, "--out-dir", OUT_DIR,
        "--sha-sample", str(SHA_SAMPLE)]
if SYMBOLS: args += ["--symbols", SYMBOLS]
if MAX_PARTITIONS: args += ["--max-partitions", str(MAX_PARTITIONS)]
if DEEP_VERIFY: args += ["--deep-verify"]
if PUBLISH_TO_DRIVE: args += ["--publish-to-drive"]
env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src")
rc = subprocess.run(args, cwd=PROJECT_DIR, env=env).returncode
print("exit code:", rc)
if PUBLISH_TO_DRIVE:
    drive.flush_and_unmount()   # Drive FUSE 為非同步上傳：不 flush 可能留下 0-byte 檔
    print("Drive flushed & unmounted")

In [ ]:
# 6. 顯示摘要
import glob, json
from IPython.display import Markdown, display
latest = sorted(glob.glob(f"{OUT_DIR}/audit_*"))[-1]
print(latest)
display(Markdown(open(f"{latest}/audit_summary.md", encoding="utf-8").read()))

In [ ]:
# 7. 打包下載（選用）：把整個 audit 資料夾壓成 zip
import shutil
zip_path = shutil.make_archive(latest, "zip", latest)
print(zip_path)
from google.colab import files
files.download(zip_path)